<a href="https://colab.research.google.com/github/Arif0000/Pytorch/blob/main/pytorch_rnn_based_QA_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()


,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
def tokenize(text):
  text = text.lower()
  text = text.replace('?', '')
  text = text.replace("'", "")
  return text.split()

In [5]:
tokenize('what is the capital of india?')

['what', 'is', 'the', 'capital', 'of', 'india']

In [6]:
vocab = {'<UNK>':0}

In [7]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [8]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [9]:
len(vocab)

324

In [10]:
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [11]:
text_to_indices("what is comb", vocab)

[1, 2, 0]

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

In [13]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]
  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [14]:
dataset = QADataset(df, vocab)

In [16]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [17]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[10, 55,  3, 56,  5, 57]]) tensor([58])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([36])
tensor([[10, 75, 76]]) tensor([77])
tensor([[ 10,  11, 189, 158, 190]]) tensor([191])
tensor([[10, 96,  3, 97]]) tensor([98])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([173])
tensor([[ 10, 308,   3, 309, 310]]) tensor([311])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([114])
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([220])
tensor([[ 42,   2,   3, 210, 137, 168, 211, 169]]) tensor([113])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([124])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([215])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([179])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([68])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([280])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([128])
tensor([[ 42, 137,   2, 138,  39

In [18]:
import torch.nn as nn


In [20]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50,64,batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output


In [21]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50,64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print('shape of a', a.shape)
b = x(a)
print('shape of b', b.shape)
c,d = y(b)
print('shape of c', c.shape)
print('shape of c', d.shape)

e = z(d.squeeze(0))
print('shape of e', e.shape)

shape of a torch.Size([1, 6])
shape of b torch.Size([1, 6, 50])
shape of c torch.Size([1, 6, 64])
shape of c torch.Size([1, 1, 64])
shape of e torch.Size([1, 324])


In [22]:
learning_rate = 0.001
epochs = 20

In [23]:
model = SimpleRNN(len(vocab))

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [25]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.050972
Epoch: 2, Loss: 459.847857
Epoch: 3, Loss: 382.238335
Epoch: 4, Loss: 318.484805
Epoch: 5, Loss: 266.520452
Epoch: 6, Loss: 217.867596
Epoch: 7, Loss: 173.639515
Epoch: 8, Loss: 134.459863
Epoch: 9, Loss: 102.386398
Epoch: 10, Loss: 78.285383
Epoch: 11, Loss: 59.630611
Epoch: 12, Loss: 46.642939
Epoch: 13, Loss: 37.029542
Epoch: 14, Loss: 29.723462
Epoch: 15, Loss: 24.666287
Epoch: 16, Loss: 20.732847
Epoch: 17, Loss: 17.506359
Epoch: 18, Loss: 15.063339
Epoch: 19, Loss: 13.160747
Epoch: 20, Loss: 11.515417


In [26]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [27]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [29]:
list(vocab.keys())[4]

'capital'